예제1

In [3]:
def deco(func):
  def inner():
    print("running inner()")
  return inner
@deco
def target():
  print("Running target()")
target()

running inner()


In [4]:
target

<function __main__.deco.<locals>.inner()>

예제1-1. Decorate예제

In [5]:
def logger(func):
  def wrapper(*args, **kwargs):
    print(f"[LOG] {func.__name__} called with args={args}, kwargs={kwargs}")
    result = func(*args, **kwargs)
    print(f"[LOG] {func.__name__} returned {result}")
    return result
  return wrapper

  @logger
  def add(a,b):
    return a+b

예제2

In [7]:
registry = []

def register(func):
  print("running register(%s)" %func)
  registry.append(func)
  return func
@register
def f1():
  print('running f1()')
@register
def f2():
  print('running f2()')
def f3():
  print('running f3()')
def main():
  print('running main()')
  print('registry ->', registry)
  f1()
  f2()
  f3()

if __name__ =='__main__':
  main()

running register(<function f1 at 0x7f047950afc0>)
running register(<function f2 at 0x7f047950ade0>)
running main()
registry -> [<function f1 at 0x7f047950afc0>, <function f2 at 0x7f047950ade0>]
running f1()
running f2()
running f3()


예제3

In [10]:
promos = []

def promotion(promo_func):
  promos.append(promo_func)
  return promo_func
@promotion
def fidelity_promo(order):
  return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

@promotion
def bulk_item_promo(order):
  discount = 0
  for item in order.cart:
    if item.quantity >= 20:
      discount += item.total() * 0.1
  return discount

@promotion
def large_order_promo(order) :
  distinct_items = {item.product for item in order.cart}
  if len(distinct_items) >= 10:
    return order.total() * 0.07
  return 0

def best_promo(order):
  return max(promo(order) for promo in promos)

예제4

In [11]:
def f1(a):
  print(a)
  print(b)
f1(3)

3


NameError: name 'b' is not defined

In [12]:
b = 6
f1(3)

3
6


예제5

In [13]:
b=6
def f2(a):
  print(a)
  print(b)
  b = 9
f2(3)

3


UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

예제6

In [15]:
b=6
def f3(a):
  global b
  print(a)
  print(b)
  b=9
f3(3)

3
6


In [17]:
f3(3)

3
9


In [19]:
b=30
f3(3)

3
30


예제7

In [25]:
class Averager():
  def __init__(self):
    self.series = []
  def __call__(self, new_value):
    self.series.append(new_value)
    total = sum(self.series)
    return total/len(self.series)

In [28]:
avg = Averager()
avg(10)

10.0

In [30]:
avg(11)

10.666666666666666

In [32]:
avg(12)

11.2

In [39]:
def make_averager():
  series = []
  def averager(new_value):
    series.append(new_value)
    total = sum(series)
    return total/len(series)
  return averager

In [40]:
avg = make_averager()
avg(10)

10.0

In [45]:
avg(11)

9.166666666666666

In [46]:
avg(12)

9.571428571428571

예제9

In [48]:
avg.__code__.co_varnames

('new_value', 'total')

In [50]:
avg.__code__.co_freevars

('series',)

In [52]:
avg.__closure__

(<cell at 0x7f0490365120: list object at 0x7f0478f8d640>,)

In [54]:
avg.__closure__[0].cell_contents

[10, 1, 11, 11, 11, 11, 12]

예제10

In [56]:
def make_averager():
  count = 0
  total = 0

  def averager(new_value) :
    count += 1
    total += new_value
    return total / count
  return averager

In [57]:
avg = make_averager()
avg(10)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

예제11

In [58]:
def make_averager():
  count = 0
  total = 0

  def averager(new_value) :
    nonlocal count, total
    count += 1
    total += new_value
    return total / count
  return averager

In [60]:
avg = make_averager()
avg(10)

10.0

예제12

In [62]:
import time

def clock(func):
  def clocked(*args):
    t0 = time.perf_counter()
    result = func(*args)
    elapsed = time.perf_counter() - t0
    name = func.__name__
    arg_str = ', '.join(repr(arg) for arg in args)
    print("[%0.8fs] %s(%s) -> %r" %(elapsed, name, arg_str, result))
    return result
  return clocked

예제13

In [64]:
@clock
def snooze(seconds):
  time.sleep(seconds)
@clock
def factorial(n):
  return 1 if n < 2 else n * factorial(n-1)

if __name__=='__main__' :
  print("*" * 40, "Calling snooze(.123)")
  snooze(.123)
  print("*"*40, 'Calling factorial(6)')
  print("6! =", factorial(6))

**************************************** Calling snooze(.123)
[0.12317002s] snooze(0.123) -> None
**************************************** Calling factorial(6)
[0.00000028s] factorial(1) -> 1
[0.00000948s] factorial(2) -> 2
[0.00001415s] factorial(3) -> 6
[0.00001864s] factorial(4) -> 24
[0.00002459s] factorial(5) -> 120
[0.00003448s] factorial(6) -> 720
6! = 720


예제14

In [65]:
import time
import functools

def clock(func):
  @functools.wraps(func)
  def clocked(*args,**kwargs):
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time()-t0
    arg_lst = []
    name = func.__name__
    if args:
      arg_lst.append(','.join(repr(arg) for arg in args))
    if kwargs :
      pairs = ["%s=%r" %(k,w) for k,w in sorted(kwargs.items())]
      arg_lst.append(', '.join(pairs))
    arg_str = ', '.join(arg_lst)
    print("[%0.8fs] %s(%s) -> %r" %(elapsed, name, arg_str, result))
    return result
  return clocked

In [67]:
@clock
def snooze(seconds) :
  time.sleep(seconds)
@clock
def factorial(n) :
  return 1 if n < 2 else n * factorial(n-1)
if __name__ == "__main__" :
  print("*" * 40, "Calling snooze(.123)")
  snooze(.123)
  print("*" * 40, "Calling factorial(6)")
  print('6! =', factorial(6))

**************************************** Calling snooze(.123)
[0.12312174s] snooze(0.123) -> None
**************************************** Calling factorial(6)
[0.00000119s] factorial(1) -> 1
[0.00001287s] factorial(2) -> 2
[0.00001884s] factorial(3) -> 6
[0.00002384s] factorial(4) -> 24
[0.00003004s] factorial(5) -> 120
[0.00003791s] factorial(6) -> 720
6! = 720
